In [1]:
print("3")

3


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

In [3]:
df = pd.read_csv("olist_labeled_ml_table_new.csv")

print("Shape:", df.shape)
df.head()

Shape: (96476, 38)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,multi_seller,freight_ratio,freight_per_item,total_weight,total_volume,avg_volume,max_volume,distance_km,is_cross_state,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,0,0.290764,8.72,500.0,1976.0,1976.0,1976.0,18.576110,0,0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,...,0,0.191744,22.76,400.0,4693.0,4693.0,4693.0,851.495069,1,0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,0,0.120200,19.22,420.0,9576.0,9576.0,9576.0,514.410666,1,0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,7c142cf63193a1473d2e66489a9ae977,59296,...,0,0.604444,27.20,450.0,6000.0,6000.0,6000.0,1822.226336,1,0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,0,0.438191,8.72,250.0,11475.0,11475.0,11475.0,29.676625,0,0


In [4]:
print("Missing labels:")
print(df["is_late"].isna().sum())

print("\nDuplicate orders:")
print(df["order_id"].duplicated().sum())

print("\nLabel distribution:")
print(df["is_late"].value_counts())

Missing labels:
0

Duplicate orders:
0

Label distribution:
is_late
0    88649
1     7827
Name: count, dtype: int64


In [5]:
df["order_purchase_timestamp"] = pd.to_datetime(
    df["order_purchase_timestamp"],
    errors="coerce"
)

print("Minimum purchase date:",
      df["order_purchase_timestamp"].min())

print("Maximum purchase date:",
      df["order_purchase_timestamp"].max())

Minimum purchase date: 2016-09-15 12:16:38
Maximum purchase date: 2018-08-29 15:00:37


In [6]:
df["purchase_year"] = df["order_purchase_timestamp"].dt.year

print(df["purchase_year"].value_counts().sort_index())

purchase_year
2016      272
2017    43426
2018    52778
Name: count, dtype: int64


In [7]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["is_late"]
)

سنستخدم هنا ال Stratified Random split لأن الهدف هنا هو تصنيف الطلبات اعتمااً على ال information available at prediction time و نريد أن نحافظ على نسبة العمود الهدف في كل split 

In [8]:
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["is_late"]
)

In [9]:
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (67533, 38)
Validation: (14471, 38)
Test: (14472, 38)


In [10]:
print(
    "\nTotal rows:",
    len(train_df) + len(val_df) + len(test_df)
)

print(
    "Original rows:",
    len(df)
)


Total rows: 96476
Original rows: 96476


In [11]:
def label_distribution(data, name):
    counts = data["is_late"].value_counts()
    percentages = data["is_late"].value_counts(normalize=True) * 100

    print(f"\n{name}")
    print("-" * 30)
    print("Rows:", len(data))
    print("On Time (0):", counts.get(0, 0),
          f"({percentages.get(0, 0):.2f}%)")
    print("Late (1):", counts.get(1, 0),
          f"({percentages.get(1, 0):.2f}%)")

In [12]:
label_distribution(df, "Full Dataset")
label_distribution(train_df, "Train")
label_distribution(val_df, "Validation")
label_distribution(test_df, "Test")


Full Dataset
------------------------------
Rows: 96476
On Time (0): 88649 (91.89%)
Late (1): 7827 (8.11%)

Train
------------------------------
Rows: 67533
On Time (0): 62054 (91.89%)
Late (1): 5479 (8.11%)

Validation
------------------------------
Rows: 14471
On Time (0): 13297 (91.89%)
Late (1): 1174 (8.11%)

Test
------------------------------
Rows: 14472
On Time (0): 13298 (91.89%)
Late (1): 1174 (8.11%)


In [13]:
print("Full dataset late ratio:",
      df["is_late"].mean())

print("Train late ratio:",
      train_df["is_late"].mean())

print("Validation late ratio:",
      val_df["is_late"].mean())

print("Test late ratio:",
      test_df["is_late"].mean())

Full dataset late ratio: 0.08112898544715784
Train late ratio: 0.08113070646942976
Validation late ratio: 0.08112777278695321
Test late ratio: 0.08112216694306247


In [14]:
train_orders = set(train_df["order_id"])
val_orders = set(val_df["order_id"])
test_orders = set(test_df["order_id"])

In [15]:
print("Train ∩ Validation:",
      len(train_orders & val_orders))

print("Train ∩ Test:",
      len(train_orders & test_orders))

print("Validation ∩ Test:",
      len(val_orders & test_orders))

Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


In [16]:
def date_range(data, name):
    print(f"\n{name}")
    print("Min:",
          data["order_purchase_timestamp"].min())
    print("Max:",
          data["order_purchase_timestamp"].max())

In [17]:
date_range(train_df, "Train")
date_range(val_df, "Validation")
date_range(test_df, "Test")


Train
Min: 2016-10-03 22:31:31
Max: 2018-08-29 15:00:37

Validation
Min: 2016-09-15 12:16:38
Max: 2018-08-29 14:52:00

Test
Min: 2016-10-03 16:56:50
Max: 2018-08-29 14:18:23


In [18]:
for data in [train_df, val_df, test_df]:
    if "purchase_year" in data.columns:
        data.drop(columns=["purchase_year"], inplace=True)

In [20]:
train_df.to_csv("train_new.csv", index=False)
val_df.to_csv("validation_new.csv", index=False)
test_df.to_csv("test.csv_new", index=False)

print("Files saved successfully.")

Files saved successfully.


In [21]:
print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain labels:")
print(train_df["is_late"].value_counts(normalize=True))

print("\nValidation labels:")
print(val_df["is_late"].value_counts(normalize=True))

print("\nTest labels:")
print(test_df["is_late"].value_counts(normalize=True))

Train shape: (67533, 37)
Validation shape: (14471, 37)
Test shape: (14472, 37)

Train labels:
is_late
0    0.918869
1    0.081131
Name: proportion, dtype: float64

Validation labels:
is_late
0    0.918872
1    0.081128
Name: proportion, dtype: float64

Test labels:
is_late
0    0.918878
1    0.081122
Name: proportion, dtype: float64
